# Capstone — Which pages will lose clicks next? A time-forward test of a content-refresh queue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matiasdibe23-sys/Flyrank-Starter-Notebooks/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring. **Output:** a ranked review queue with reason codes, plus an honest test of whether a model beats a one-line rule.

**How to run:** add a Colab secret named `HF_TOKEN` (read token, access to `FlyRank/internship-warehouse` already approved), then *Runtime → Run all*. The first run reads six months of daily data from Hugging Face and caches one small table per month, so allow several minutes. Seeds are fixed, so a re-run reproduces the numbers.

**Public-safety:** no client names, URLs or queries are used or printed. Hashed ids are used only to group and join; they are never printed and never used as features. Only aggregates and anonymous example rows leave this notebook.

## 1. Question

**Decision supported.** A content team can refresh only a small share of its pages each cycle. Given what a page did in the **first half of a month**, which pages should be reviewed first because they are likely to lose clicks in the **second half**?

**Unit.** One page (content item) in one calendar month. **Lane:** Refresh / Content Opportunity Scoring. **Output:** a ranked review queue with validated reason codes.

**Why this question.** The first modelling attempt in this repo (W05 v1) reached 1.0000 on every metric because the label was a SQL rule over its own features — leakage by construction (`work/outputs/w05_audit_report.md`). The rebuilt question below has a label that comes from a later time window, a baseline that is allowed to win, and a test on months the model never saw.

## 2. Data

Source: the FlyRank ML Internship warehouse release (`fact_content_daily_performance`, daily search performance per page, Hive-partitioned by month). Windows: January–June 2026; training on January–April, testing on May and June. Observable search signals only (impressions, clicks, average position) — no product flags or scores. The daily table has about 79 million rows; each month is aggregated to one row per page **inside DuckDB** and cached, so only small tables reach pandas. Exclusions: rows without Search Console data, pages below a traffic floor (too few clicks to measure a decline), pages with fewer than 7 days of data in either half of the month.

In [ ]:
import os, sys, json, calendar, importlib, subprocess, warnings
from datetime import datetime

for _pkg in ["duckdb"]:
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", _pkg])

import numpy as np
import pandas as pd
import duckdb
import sklearn
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------- configuration (every knob in one place) ----------------
REL = os.environ.get("CAPSTONE_REL", "hf://datasets/FlyRank/internship-warehouse")
FACT_GLOB = REL + "/fact_content_daily_performance/month={m}/*.parquet"

TRAIN_MONTHS = ["2026-01", "2026-02", "2026-03", "2026-04"]
TEST_MONTHS = ["2026-05", "2026-06"]
ALL_MONTHS = TRAIN_MONTHS + TEST_MONTHS

PRIMARY_FLOOR = 50           # min clicks in days 1-15 (sensitivity: 20 and 100)
SENS_FLOORS = [20, 50, 100]
MIN_DAYS = 7                 # min days with data in EACH half
DECAY_THRESHOLD = 0.80       # "more than 20% worse than the typical page"
HELDOUT_CLIENT_FRAC = 0.20
SEED = 42
N_BOOT = 200                 # client-cluster bootstrap repetitions
TOP_FRAC = 0.10              # review budget: top 10% of pages
CACHE_DIR = "capstone_cache"
OUT_DIR = "work/outputs"
FIG_DIR = "work/figures"
for d in (CACHE_DIR, OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

con = duckdb.connect()
if REL.startswith("hf://"):
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        raise RuntimeError("Add a Colab secret named HF_TOKEN (read token) and re-run.")
    con.execute("INSTALL httpfs; LOAD httpfs;")
    try:
        con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")
    except Exception as e:
        # print only the exception type: the message could echo SQL text that contains the token
        raise RuntimeError(f"Could not create the Hugging Face secret ({type(e).__name__}).") from None
print(f"pandas {pd.__version__} | scikit-learn {sklearn.__version__} | duckdb {duckdb.__version__} | seed {SEED}")

### Aggregation and coverage

One row per page and month, aggregated in DuckDB, cached locally. The coverage table shows how much data each month has: months with thin coverage would make the train/test comparison fragile.

In [ ]:
def month_sql(m, min_clicks):
    nd = calendar.monthrange(int(m[:4]), int(m[5:]))[1]
    d1, w1e, w2s, h1e, h2s, end = (f"{m}-01", f"{m}-07", f"{m}-08", f"{m}-15", f"{m}-16", f"{m}-{nd:02d}")
    path = FACT_GLOB.format(m=m)
    sql = f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{h1e}') AS impr_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '{h1e}') AS clicks_h1,
        COUNT(*)             FILTER (WHERE report_date <= DATE '{h1e}') AS days_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '{w1e}')
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{w1e}'), 0) AS pos_w1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{w2s}' AND DATE '{h1e}')
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{w2s}' AND DATE '{h1e}'), 0) AS pos_w2,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '{w1e}') AS clicks_w1,
        COUNT(*)             FILTER (WHERE report_date <= DATE '{w1e}') AS days_w1,
        SUM(gsc_clicks)      FILTER (WHERE report_date BETWEEN DATE '{w2s}' AND DATE '{h1e}') AS clicks_w2,
        COUNT(*)             FILTER (WHERE report_date BETWEEN DATE '{w2s}' AND DATE '{h1e}') AS days_w2,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{w1e}') AS impr_w1,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{w2s}' AND DATE '{h1e}') AS impr_w2,
        -- label source: second half only (never a feature)
        SUM(gsc_clicks)      FILTER (WHERE report_date >= DATE '{h2s}') AS clicks_h2,
        COUNT(*)             FILTER (WHERE report_date >= DATE '{h2s}') AS days_h2
    FROM read_parquet('{path}')
    WHERE report_date BETWEEN DATE '{d1}' AND DATE '{end}' AND {AVAIL}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '{h1e}') >= {min_clicks}
    """
    return sql, nd

_cols = {r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT_GLOB.format(m=ALL_MONTHS[0])}')").fetchall()}
_missing = {"report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks", "gsc_avg_position"} - _cols
assert not _missing, f"columns not found in the fact table: {_missing}"
AVAIL = "gsc_data_available IS TRUE" if "gsc_data_available" in _cols else "gsc_clicks IS NOT NULL"
print("Row filter for Search Console data:", AVAIL)

def load_month(m):
    cache = os.path.join(CACHE_DIR, f"month_{m}_floor{min(SENS_FLOORS)}.parquet")
    if os.path.exists(cache):
        return pd.read_parquet(cache)
    sql, nd = month_sql(m, min(SENS_FLOORS))
    d = con.execute(sql).df()
    d["month"] = m
    d["n_days_h2_cal"] = nd - 15          # calendar days in the second half (15, 15 or 16, 13...)
    d.to_parquet(cache)
    return d

raw = pd.concat([load_month(m) for m in ALL_MONTHS], ignore_index=True)
raw["client_hash_id"] = raw.client_hash_id.fillna("unknown")

cov = (raw.groupby("month").agg(pages_ge_floor20=("content_hash_id", "size"), clients=("client_hash_id", "nunique"))
          .reindex(ALL_MONTHS))
print("=== COVERAGE: pages with >= 20 clicks in days 1-15, by month ===")
print(cov.to_string())
missing = [m for m in ALL_MONTHS if not (cov.loc[m, "pages_ge_floor20"] > 0)]
assert not missing, f"no data for months {missing}: edit TRAIN_MONTHS / TEST_MONTHS"

## 3. Methodology

**Design, and why it is honest for this question**

1. *Within each month,* features come only from days 1–15 and the label only from days 16–end. No feature is computed from the label window (checked by an assertion below).
2. *Across months,* the model trains on **January–April 2026** and is tested on **May and June 2026**. This is a time-forward test: the model never sees the months it is graded on.
3. *Across clients,* clients are split once (80/20, fixed seed). **Test A** = later months, clients seen in training. **Test B** = later months, clients the model has never seen (the harder, more realistic case for a new customer).
4. *Label.* A page "decays" if its clicks/day in the second half fell more than 20% **more than the typical page's** in the same month (market-adjusted). The raw label (`H2 < 0.8 × H1`) is kept as a sensitivity check. Reason: the first W05 run showed that most pages fall in the second half of June for reasons shared by all pages (mean reversion, a market-wide shift), which a raw threshold mistakes for page decay.
5. *Baselines on the same split:* a click-momentum rule, the Week-4 opportunity score, and size only. A model is only interesting if it beats these.

**Choices made after seeing June (disclosure).** The traffic floor (50 clicks in days 1–15) and the lean feature set (no pageviews, no raw clicks, no days-with-data) were chosen after the single-month W05 analysis, which used June. June is therefore *not a blind test*; **May is the cleaner test month**, and every table reports the two months separately.

### Features and label (features: days 1–15 only; label: days 16–end only)

| Feature | Meaning |
|---|---|
| `impr_h1` | impressions, days 1–15 |
| `ctr_h1` | clicks / impressions, days 1–15 |
| `pos_h1` | impression-weighted average position, days 1–15 |
| `click_momentum` | clicks/day in days 8–15 relative to days 1–7 (below 1 means already falling) |
| `impr_momentum` | the same ratio for impressions |
| `pos_slip` | position in days 8–15 minus position in days 1–7 (positive means sliding down the results) |

`clicks_h1` is used for eligibility, for the size-only baseline and for the impact metric, but it is not a model feature.

In [ ]:
LEAN = ["impr_h1", "ctr_h1", "pos_h1", "click_momentum", "impr_momentum", "pos_slip"]
LABEL_WINDOW_COLS = {"clicks_h2", "days_h2", "rate_h2", "ratio", "y_adj", "y_raw", "lost_adj", "lost_raw", "n_days_h2_cal"}
assert LABEL_WINDOW_COLS.isdisjoint(LEAN), "a label-window column leaked into the feature list"

d = raw[(raw.days_h1 >= MIN_DAYS) & (raw.days_h2 >= MIN_DAYS)].copy()
d["pos_h1"] = (d.pos_w1.fillna(d.pos_w2) + d.pos_w2.fillna(d.pos_w1)) / 2
d["ctr_h1"] = d.clicks_h1 / d.impr_h1.replace(0, np.nan)
rc1 = d.clicks_w1.fillna(0) / d.days_w1.replace(0, np.nan)
rc2 = d.clicks_w2.fillna(0) / d.days_w2.replace(0, np.nan)
d["click_momentum"] = ((rc2 + 1) / (rc1 + 1)).fillna(1.0)
ri1 = d.impr_w1.fillna(0) / d.days_w1.replace(0, np.nan)
ri2 = d.impr_w2.fillna(0) / d.days_w2.replace(0, np.nan)
d["impr_momentum"] = ((ri2 + 1) / (ri1 + 1)).fillna(1.0)
d["pos_slip"] = (d.pos_w2 - d.pos_w1).fillna(0.0)
d["rate_h1"] = d.clicks_h1 / d.days_h1
d["rate_h2"] = d.clicks_h2 / d.days_h2
d["ratio"] = d.rate_h2 / d.rate_h1

def label_for_floor(frame, floor):
    """Apply a traffic floor, then build both labels. The cohort median is computed per month, so the
    market-wide shift of that month is removed from the label (the label may use second-half data; features never do)."""
    f = frame[frame.clicks_h1 >= floor].copy()
    med = f.groupby("month").ratio.transform("median")
    f["cohort_ratio"] = med
    f["y_adj"] = (f.ratio < DECAY_THRESHOLD * med).astype(int)
    f["y_raw"] = (f.ratio < DECAY_THRESHOLD).astype(int)
    f["lost_adj"] = (f.rate_h1 * med - f.rate_h2).clip(lower=0) * f.n_days_h2_cal   # estimated clicks lost beyond the market move
    f["lost_raw"] = (f.rate_h1 - f.rate_h2).clip(lower=0) * f.n_days_h2_cal
    return f.dropna(subset=LEAN + ["rate_h1", "rate_h2"]).reset_index(drop=True)

data = label_for_floor(d, PRIMARY_FLOOR)
summ = data.groupby("month").agg(pages=("y_adj", "size"), clients=("client_hash_id", "nunique"),
                                 decay_rate_adj=("y_adj", "mean"), decay_rate_raw=("y_raw", "mean"),
                                 median_page_ratio=("cohort_ratio", "first")).reindex(ALL_MONTHS).round(3)
print(f"=== ANALYSIS TABLE (floor = {PRIMARY_FLOOR} clicks in days 1-15, >= {MIN_DAYS} days with data per half) ===")
print(summ.to_string())
print("\nA median page ratio below 1 means the typical page lost clicks in the second half; the adjusted label ignores that shared move.")

### Split: time-forward and client-aware

Clients are split once into a training group and a held-out group. Training uses January–April for training clients. **Test A:** May–June, training clients. **Test B:** May–June, held-out clients the model has never seen.

In [ ]:
rng = np.random.RandomState(SEED)
all_clients = np.array(sorted(data.client_hash_id.unique()))
heldout = set(rng.permutation(all_clients)[: max(1, int(round(HELDOUT_CLIENT_FRAC * len(all_clients))))])

def split(frame):
    is_h = frame.client_hash_id.isin(heldout)
    train = frame[frame.month.isin(TRAIN_MONTHS) & ~is_h]
    test_a = frame[frame.month.isin(TEST_MONTHS) & ~is_h]
    test_b = frame[frame.month.isin(TEST_MONTHS) & is_h]
    return train, test_a, test_b

train, test_a, test_b = split(data)
assert set(train.client_hash_id).isdisjoint(set(test_b.client_hash_id)), "a held-out client is in training"
assert train.month.max() < test_a.month.min(), "training months must precede test months"
print(f"Train: {len(train):,} pages / {train.client_hash_id.nunique()} clients ({', '.join(TRAIN_MONTHS)})")
print(f"Test A (later months, seen clients):   {len(test_a):,} pages / {test_a.client_hash_id.nunique()} clients")
print(f"Test B (later months, unseen clients): {len(test_b):,} pages / {test_b.client_hash_id.nunique()} clients")
print(f"Decay rate (adjusted label) -> train {train.y_adj.mean():.3f} | test A {test_a.y_adj.mean():.3f} | test B {test_b.y_adj.mean():.3f}")

## 4. Results (vs baseline)

### Models and baselines (same split, same metrics)

- **Momentum rule** — rank by lowest `click_momentum`. No training.
- **Week-4 score** — the Week-4 opportunity heuristic (impressions, CTR, position). It measures *upside*, not *downside risk*; it is here to check that it should not be used for this decision.
- **Size only** — rank by clicks in days 1–15.
- **Logistic regression, Random Forest, gradient boosting** — trained on the lean features, January–April.
- **× size variants** — `P(decay) × clicks` (and `momentum risk × clicks`): a refresh queue should prefer pages where a drop costs more clicks.

Metrics: ROC-AUC and PR-AUC (read against the base rate), precision and lift at the top 10%, and **lost clicks captured at 10%** — if the team can review only 10% of pages, what share of the clicks lost beyond the market move sits inside that selection. Random selection captures about 10%.

In [ ]:
COUNT_COLS = ["impr_h1"]

def _log(frame):
    f = frame.copy(); f[COUNT_COLS] = np.log1p(f[COUNT_COLS]); return f

from sklearn.preprocessing import FunctionTransformer

def w04_score(f):
    return np.log1p(f.impr_h1) * 10.0 + (1.0 - f.ctr_h1.clip(0, 1)) * 40.0 + np.where(f.pos_h1 <= 15, 20.0, 0.0)

def fit_models(train_df, label):
    X, y = train_df[LEAN], train_df[label]
    return {
        "lr": make_pipeline(FunctionTransformer(_log), StandardScaler(), LogisticRegression(max_iter=1000)).fit(X, y),
        "rf": RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=50, n_jobs=-1, random_state=SEED).fit(X, y),
        "hgb": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=200, min_samples_leaf=100, random_state=SEED).fit(X, y),
    }

def score_all(models, f):
    p_rf = models["rf"].predict_proba(f[LEAN])[:, 1]
    mom = (-f.click_momentum).to_numpy()
    return {
        "Momentum rule": mom,
        "Week-4 score": np.asarray(w04_score(f)),
        "Size only (H1 clicks)": f.clicks_h1.to_numpy(),
        "Logistic regression": models["lr"].predict_proba(f[LEAN])[:, 1],
        "Random Forest": p_rf,
        "Gradient boosting": models["hgb"].predict_proba(f[LEAN])[:, 1],
        "Momentum risk x size": np.clip(1 - f.click_momentum, 0, None).to_numpy() * f.clicks_h1.to_numpy(),
        "RF P(decay) x size": p_rf * f.clicks_h1.to_numpy(),
    }

def rank_metrics(y_true, s, lost, top_frac=TOP_FRAC):
    y_true, s, lost = np.asarray(y_true), np.asarray(s), np.asarray(lost)
    k = max(1, int(len(y_true) * top_frac))
    top = np.argsort(-s, kind="stable")[:k]
    p = y_true[top].mean()
    return {"ROC-AUC": roc_auc_score(y_true, s), "PR-AUC": average_precision_score(y_true, s),
            "Precision@10%": p, "Lift@10%": p / y_true.mean(), "Lost clicks captured@10%": lost[top].sum() / max(lost.sum(), 1e-9)}

def eval_set(models, f, label="y_adj", lostcol="lost_adj"):
    sc = score_all(models, f)
    rows = []
    for name, s in sc.items():
        m = rank_metrics(f[label], s, f[lostcol]); m["Scorer"] = name; rows.append(m)
    return pd.DataFrame(rows).set_index("Scorer"), sc

models = fit_models(train, "y_adj")
results = {}
for tag, f in (("Test A", test_a), ("Test B", test_b)):
    for m in TEST_MONTHS:
        fm = f[f.month == m]
        if len(fm) < 200 or fm.y_adj.nunique() < 2:
            print(f"[skip] {tag} {m}: only {len(fm)} pages"); continue
        tbl, _ = eval_set(models, fm)
        results[(tag, m)] = tbl
        print(f"\n=== {tag} | {m} | pages={len(fm):,} | base rate (decay) = {fm.y_adj.mean():.3f} ===")
        print(tbl.round(4).to_string())

### How sure are we? Client-cluster bootstrap

Pages of one client move together, so resampling pages would overstate certainty. The bootstrap below resamples **whole clients** (200 repetitions) and reports a 95% interval for each scorer's ROC-AUC and for the difference to the momentum rule. If an interval for a difference contains 0, the data cannot separate the two.

In [ ]:
def cluster_bootstrap(f, models, n_boot=N_BOOT, seed=SEED):
    sc = score_all(models, f)
    y = f.y_adj.to_numpy()
    cl = f.client_hash_id.to_numpy()
    idx_by_client = {c: np.where(cl == c)[0] for c in np.unique(cl)}
    clients = np.array(list(idx_by_client))
    rg = np.random.RandomState(seed)
    out = {k: [] for k in sc}
    for _ in range(n_boot):
        pick = rg.choice(clients, size=len(clients), replace=True)
        idx = np.concatenate([idx_by_client[c] for c in pick])
        if len(np.unique(y[idx])) < 2:
            continue
        for k, s in sc.items():
            out[k].append(roc_auc_score(y[idx], s[idx]))
    point = {k: roc_auc_score(y, s) for k, s in sc.items()}
    rows = []
    for k in sc:
        arr = np.array(out[k]); base = np.array(out["Momentum rule"])
        diff = arr - base
        rows.append({"Scorer": k, "ROC-AUC": point[k], "CI low": np.percentile(arr, 2.5), "CI high": np.percentile(arr, 97.5),
                     "vs momentum": point[k] - point["Momentum rule"],
                     "diff CI low": np.percentile(diff, 2.5), "diff CI high": np.percentile(diff, 97.5)})
    return pd.DataFrame(rows).set_index("Scorer"), len(out["Momentum rule"])

boot = {}
for tag, f in (("Test A", test_a), ("Test B", test_b)):
    if len(f) and f.y_adj.nunique() == 2 and f.client_hash_id.nunique() >= 3:
        boot[tag], nb = cluster_bootstrap(f, models)
        print(f"\n=== {tag}: pooled May+June, {f.client_hash_id.nunique()} clients, {nb} valid bootstrap draws ===")
        print(boot[tag].round(3).to_string())
    else:
        print(f"[skip] {tag}: too few clients for a cluster bootstrap")

# Sanity: shuffled labels must collapse to ~0.5
if len(test_a):
    ps = models["rf"].predict_proba(test_a[LEAN])[:, 1]
    shuffled_auc = roc_auc_score(np.random.RandomState(0).permutation(test_a.y_adj.to_numpy()), ps)
    print(f"\nSanity check: RF score vs shuffled labels -> ROC-AUC {shuffled_auc:.3f} (expect ~0.5)")

### Sensitivity: does the conclusion depend on my choices?

Three traffic floors (20, 50, 100 clicks in days 1–15) and two labels (market-adjusted vs raw), each retrained from scratch and evaluated on Test A and Test B pooled. If the ranking of scorers changes with a choice, that choice matters and the paper must say so.

In [ ]:
sens_rows = []
for floor in SENS_FLOORS:
    dd = label_for_floor(d, floor)
    tr, ta, tb = split(dd)
    for label, lostcol in (("y_adj", "lost_adj"), ("y_raw", "lost_raw")):
        if tr[label].nunique() < 2:
            continue
        ms = fit_models(tr, label)
        for tag, f in (("Test A", ta), ("Test B", tb)):
            if len(f) < 200 or f[label].nunique() < 2:
                continue
            tbl, _ = eval_set(ms, f, label, lostcol)
            for scorer in ("Momentum rule", "Random Forest", "Gradient boosting", "Week-4 score", "Size only (H1 clicks)", "RF P(decay) x size"):
                sens_rows.append({"floor": floor, "label": "adjusted" if label == "y_adj" else "raw", "set": tag, "pages": len(f),
                                  "base rate": f[label].mean(), "Scorer": scorer,
                                  "ROC-AUC": tbl.loc[scorer, "ROC-AUC"], "Lost clicks captured@10%": tbl.loc[scorer, "Lost clicks captured@10%"]})
sens = pd.DataFrame(sens_rows)
pv = sens.pivot_table(index=["set", "label", "floor", "pages", "base rate"], columns="Scorer", values="ROC-AUC").round(3)
print("=== SENSITIVITY: ROC-AUC by floor / label / set ===")
print(pv.to_string())

### What the model leans on

Permutation importance: the drop in ROC-AUC when one feature is shuffled, on Test A (held back from training).

In [ ]:
samp = test_a.sample(n=min(20000, len(test_a)), random_state=SEED) if len(test_a) else test_a
if len(samp) and samp.y_adj.nunique() == 2:
    perm = permutation_importance(models["rf"], samp[LEAN], samp.y_adj, scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
    imp = pd.DataFrame({"Feature": LEAN, "AUC drop": perm.importances_mean, "std": perm.importances_std}).sort_values("AUC drop", ascending=False)
    print(imp.round(4).to_string(index=False))
else:
    imp = pd.DataFrame(columns=["Feature", "AUC drop", "std"])

### Leakage negative control (the harness must be able to *detect* a leak)

A test harness that never fails proves nothing. I add label-window information as a feature **on purpose** and check that the score jumps. Two controls:

1. `ratio` (second-half clicks/day over first-half clicks/day). The label is `ratio < 0.8 x (that month's median ratio)`, and the model does not know the month's median, so this leak is strong but not exact; the score should rise far above the honest model.
2. `ratio_vs_cohort` = `ratio / median ratio of the month`, which *is* the label's own quantity. This is the exact leak and must score about 1.0.

The honest model is shown next to them.

In [ ]:
LEAKY1 = LEAN + ["ratio"]
LEAKY2 = LEAN + ["ratio_vs_cohort"]
tr_l, ta_l = train.assign(ratio_vs_cohort=train.ratio / train.cohort_ratio), test_a.assign(ratio_vs_cohort=test_a.ratio / test_a.cohort_ratio)
def _leaky_auc(cols):
    m = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=50, n_jobs=-1, random_state=SEED).fit(tr_l[cols], tr_l.y_adj)
    return roc_auc_score(ta_l.y_adj, m.predict_proba(ta_l[cols])[:, 1])
auc_honest = roc_auc_score(test_a.y_adj, models["rf"].predict_proba(test_a[LEAN])[:, 1])
auc_leaky = _leaky_auc(LEAKY1)
auc_leaky_exact = _leaky_auc(LEAKY2)
print(f"Test A ROC-AUC, honest Random Forest:                         {auc_honest:.3f}")
print(f"Test A ROC-AUC, + label-window ratio (strong leak):            {auc_leaky:.3f}")
print(f"Test A ROC-AUC, + ratio / month median (exact label leak):     {auc_leaky_exact:.3f}")
assert auc_leaky > auc_honest + 0.15, "negative control failed: a strong deliberate leak did not raise the score"
assert auc_leaky_exact > 0.99, "negative control failed: the exact label leak did not reach ~1.0"


## 5. Limitations

What this work **cannot** claim:

- **No causal claim.** Data are observational. A page flagged for review is *associated* with later click loss; nothing here shows that refreshing it would recover clicks.
- **The label is a proxy.** Losing clicks beyond the typical page is a signal that a page may need review, not a measured "needs refresh" outcome. Seasonality, SERP changes, demand shifts and tracking changes can all produce it.
- **Survivorship and outcome-window selection.** A page is kept only if it has at least 7 days of data in *both* halves of the month, so the population itself depends on the outcome window. Pages that vanish from the data are invisible to the queue. The rule removed only a small share of rows (printed below), but it is a choice, and it is disclosed here.
- **The label uses same-month information.** The market adjustment divides by the median page ratio of the *same* month, which is computed from second-half data. This is legitimate for a label (features never use it), but it means the label cannot be computed for a month in real time, only after the month ends.
- **Not a fully blind test.** The floor and feature choices were informed by June in W05 (see Methodology); May is the cleaner month.
- **Short windows, one year.** Fifteen-day windows inside single months; no multi-year seasonality; only the months present in the release.
- **Few unseen clients.** Test B contains only the held-out clients, so its intervals are wide; read it as directional.
- **Pseudonymous data.** Client and page ids are hashes used for grouping only; no client names, domains, URLs or queries were used.

The cell below prints the sizes that bound these claims.

In [ ]:
print(f"Months: train {TRAIN_MONTHS} | test {TEST_MONTHS}")
print(f"Test A: {len(test_a):,} pages / {test_a.client_hash_id.nunique()} clients | Test B: {len(test_b):,} pages / {test_b.client_hash_id.nunique()} clients")
print(f"Pages removed by the >= {MIN_DAYS}-days-per-half rule (all months, floor 20): {len(raw) - len(raw[(raw.days_h1 >= MIN_DAYS) & (raw.days_h2 >= MIN_DAYS)]):,} of {len(raw):,}")
print("Base rate of the adjusted label is by construction near 25-35% in each month; a high precision@10% must be read against it.")

## 6. Ranked recommendations

### From scores to actions: reason codes, validated

A ranked list is more useful when each page says *why*. Reason codes below are plain rules; their thresholds are taken from the **training** months only. Each code is then **tested on Test A**: do flagged pages really decay more than unflagged ones? A code that does not separate is dropped from the playbook.

| Code | Rule (thresholds from training months) | Suggested review |
|---|---|---|
| `CLICKS_FALLING` | `click_momentum` below the 25th percentile | check freshness and SERP changes for the page's queries |
| `IMPRESSIONS_FALLING` | `impr_momentum` below the 25th percentile | check demand and ranking visibility |
| `POSITION_SLIPPING` | `pos_slip` above the 75th percentile | review on-page content against competing results |
| `LOW_CTR_FOR_POSITION` | CTR in the bottom quartile of pages in the same position band | review title and description |
| `HIGH_VOLUME` | `clicks_h1` at or above the 90th percentile | flag for senior review before any edit (cost of a mistake is higher) |

In [ ]:
def position_band(p):
    return pd.cut(p, [0, 3, 10, 20, 1e9], labels=["1-3", "4-10", "11-20", "21+"]).astype(str)

thr = {
    "click_momentum_q25": train.click_momentum.quantile(0.25),
    "impr_momentum_q25": train.impr_momentum.quantile(0.25),
    "pos_slip_q75": train.pos_slip.quantile(0.75),
    "clicks_h1_q90": train.clicks_h1.quantile(0.90),
}
tb_ = train.assign(band=position_band(train.pos_h1))
ctr_q25_by_band = tb_.groupby("band").ctr_h1.quantile(0.25).to_dict()

def add_codes(f):
    f = f.copy()
    band = position_band(f.pos_h1)
    f["CLICKS_FALLING"] = f.click_momentum < thr["click_momentum_q25"]
    f["IMPRESSIONS_FALLING"] = f.impr_momentum < thr["impr_momentum_q25"]
    f["POSITION_SLIPPING"] = f.pos_slip > thr["pos_slip_q75"]
    f["LOW_CTR_FOR_POSITION"] = f.ctr_h1 < band.map(ctr_q25_by_band).astype(float)
    f["HIGH_VOLUME"] = f.clicks_h1 >= thr["clicks_h1_q90"]
    return f

CODES = ["CLICKS_FALLING", "IMPRESSIONS_FALLING", "POSITION_SLIPPING", "LOW_CTR_FOR_POSITION", "HIGH_VOLUME"]
ta_c = add_codes(test_a)
rows = []
for c in CODES:
    fl = ta_c[c]
    if fl.sum() == 0 or (~fl).sum() == 0:
        continue
    rows.append({"Reason code": c, "Share of pages flagged": fl.mean(),
                 "Decay rate, flagged": ta_c.y_adj[fl].mean(), "Decay rate, not flagged": ta_c.y_adj[~fl].mean(),
                 "Risk ratio": ta_c.y_adj[fl].mean() / max(ta_c.y_adj[~fl].mean(), 1e-9),
                 "Share of lost clicks in flagged": ta_c.lost_adj[fl].sum() / max(ta_c.lost_adj.sum(), 1e-9)})
code_validation = pd.DataFrame(rows).set_index("Reason code")
print(f"Base rate (Test A, adjusted label): {ta_c.y_adj.mean():.3f}")
print(code_validation.round(3).to_string())

### The ranked queue and its tiers

Queue score = `P(decay) × clicks in days 1–15` from the Random Forest (expected loss). Tiers by queue rank: **REVIEW_FIRST** = top 10%, **WATCHLIST** = next 20%, **NO_ACTION** = the rest. The table checks the tiers against what actually happened in the held-back months.

In [ ]:
def tier_by_rank(score):
    r = pd.Series(score).rank(ascending=False, method="first") / len(score)
    return np.where(r <= 0.10, "REVIEW_FIRST", np.where(r <= 0.30, "WATCHLIST", "NO_ACTION"))

tier_rows = []
queue_examples = None
for m in TEST_MONTHS:
    fm = ta_c[ta_c.month == m]
    if len(fm) < 200:
        continue
    fm = fm.assign(queue_score=models["rf"].predict_proba(fm[LEAN])[:, 1] * fm.clicks_h1.to_numpy())
    fm["tier"] = tier_by_rank(fm.queue_score.to_numpy())
    for t in ("REVIEW_FIRST", "WATCHLIST", "NO_ACTION"):
        g = fm[fm.tier == t]
        tier_rows.append({"month": m, "tier": t, "pages": len(g), "Decay rate": g.y_adj.mean(),
                          "Share of lost clicks": g.lost_adj.sum() / max(fm.lost_adj.sum(), 1e-9)})
    if m == TEST_MONTHS[0]:
        top = fm.sort_values("queue_score", ascending=False).head(10).copy()
        top["reason codes"] = top[CODES].apply(lambda r: ", ".join([c for c in CODES if r[c]]) or "(none)", axis=1)
        queue_examples = top.reset_index(drop=True)[["tier", "clicks_h1", "click_momentum", "pos_h1", "reason codes", "y_adj"]]
        queue_examples.index = queue_examples.index + 1
        queue_examples.index.name = "rank"
tiers = pd.DataFrame(tier_rows)
print(tiers.round(3).to_string(index=False))
if queue_examples is not None:
    print(f"\n=== ANONYMOUS EXAMPLE: top 10 of the {TEST_MONTHS[0]} queue (no ids shown; y_adj=1 means it did decay) ===")
    print(queue_examples.round(3).to_string())

## 7. Artifacts the paper embeds

### Figures and receipts

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 140, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
BLUE, GREY = "#2563EB", "#94A3B8"

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, name)); plt.close()

if "Test A" in boot:
    b = boot["Test A"].sort_values("ROC-AUC")
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    ax.barh(b.index, b["ROC-AUC"], color=[BLUE if i in ("Momentum rule", "Random Forest", "Gradient boosting") else GREY for i in b.index])
    ax.errorbar(b["ROC-AUC"], range(len(b)), xerr=[b["ROC-AUC"] - b["CI low"], b["CI high"] - b["ROC-AUC"]], fmt="none", ecolor="black", lw=1)
    ax.axvline(0.5, color="black", lw=0.8, ls="--"); ax.set_xlabel("ROC-AUC on later months, seen clients (Test A), 95% client-bootstrap interval")
    savefig("fig1_auc_test_a.png")

lc = pd.DataFrame({f"{tag} {m}": tbl["Lost clicks captured@10%"] for (tag, m), tbl in results.items()})
if len(lc):
    ax = lc.loc[["Momentum rule", "Size only (H1 clicks)", "Random Forest", "Momentum risk x size", "RF P(decay) x size"]].plot.bar(figsize=(7, 3.4), rot=20, width=0.8)
    ax.axhline(TOP_FRAC, color="black", lw=0.8, ls="--"); ax.set_ylabel("share of lost clicks in the top 10% (random ~ 10%)")
    savefig("fig2_lost_clicks_captured.png")

if len(tiers):
    pt = tiers.pivot(index="tier", columns="month", values="Decay rate").loc[["REVIEW_FIRST", "WATCHLIST", "NO_ACTION"]]
    ax = pt.plot.bar(figsize=(5.6, 3.2), rot=0); ax.axhline(ta_c.y_adj.mean(), color="black", lw=0.8, ls="--")
    ax.set_ylabel("observed decay rate"); savefig("fig3_tier_decay_rate.png")

if len(code_validation):
    ax = code_validation["Risk ratio"].sort_values().plot.barh(figsize=(5.6, 3.0), color=BLUE)
    ax.axvline(1.0, color="black", lw=0.8, ls="--"); ax.set_xlabel("risk ratio: decay rate flagged / not flagged (Test A)")
    savefig("fig4_reason_code_risk_ratio.png")

def tbl_json(t):
    return json.loads(t.reset_index().to_json(orient="records", double_precision=4)) if len(t) else []

receipt = {
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "question": "Which pages lose clicks in the second half of a month beyond the typical page, using only the first half?",
    "design": {"train_months": TRAIN_MONTHS, "test_months": TEST_MONTHS, "primary_floor": PRIMARY_FLOOR, "min_days_each_half": MIN_DAYS,
               "decay_threshold": DECAY_THRESHOLD, "heldout_client_fraction": HELDOUT_CLIENT_FRAC, "seed": SEED, "bootstrap_draws": N_BOOT,
               "features": LEAN, "label": "clicks/day in days 16-end < 0.8 x median page ratio of the same month"},
    "versions": {"pandas": pd.__version__, "scikit-learn": sklearn.__version__, "duckdb": duckdb.__version__},
    "coverage_by_month": tbl_json(cov),
    "analysis_table_by_month": tbl_json(summ),
    "n_pages": {"train": int(len(train)), "test_a": int(len(test_a)), "test_b": int(len(test_b))},
    "n_clients": {"train": int(train.client_hash_id.nunique()), "test_a": int(test_a.client_hash_id.nunique()), "test_b": int(test_b.client_hash_id.nunique()), "heldout": len(heldout)},
    "base_rate": {"train": float(train.y_adj.mean()), "test_a": float(test_a.y_adj.mean()) if len(test_a) else None, "test_b": float(test_b.y_adj.mean()) if len(test_b) else None},
    "metrics_by_set_month": {f"{k[0]} | {k[1]}": tbl_json(v) for k, v in results.items()},
    "bootstrap_auc": {k: tbl_json(v) for k, v in boot.items()},
    "shuffled_label_sanity_auc": float(shuffled_auc) if len(test_a) else None,
    "leakage_negative_control": {"honest_rf_auc_test_a": float(auc_honest), "rf_with_label_window_ratio_auc_test_a": float(auc_leaky), "rf_with_ratio_over_month_median_auc_test_a": float(auc_leaky_exact)},
    "sensitivity": tbl_json(sens),
    "permutation_importance": tbl_json(imp.set_index("Feature")) if len(imp) else [],
    "reason_code_thresholds": {k: float(v) for k, v in thr.items()} | {"ctr_q25_by_position_band": {k: float(v) for k, v in ctr_q25_by_band.items()}},
    "reason_code_validation": tbl_json(code_validation),
    "tier_validation": tbl_json(tiers),
    "limitations": [
        "observational data: associations and ranking quality, not causal effects of refreshing",
        "decay in clicks is a proxy for 'needs review', not a measured refresh outcome",
        "pages missing from the data in either half are excluded (survivorship)",
        "traffic floor and lean feature set were chosen after the single-month W05 analysis on June; May is the cleaner test month",
        "within-month windows of 15 days; no multi-month seasonality adjustment beyond the monthly market median",
    ],
}
with open(os.path.join(OUT_DIR, "capstone_results.json"), "w") as f:
    json.dump(receipt, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
if queue_examples is not None:
    queue_examples.round(4).to_csv(os.path.join(OUT_DIR, "capstone_queue_examples.csv"))
print(f"Receipts written to {OUT_DIR}/ and figures to {FIG_DIR}/")

### Auto-written reading of the results

Every sentence below is computed from the tables above, so it cannot drift from the numbers. Read it together with the limitations.

In [ ]:
def fmt(x): return f"{x:.3f}"
lines = []
for tag in ("Test A", "Test B"):
    if tag not in boot:
        continue
    b = boot[tag]
    best = b["ROC-AUC"].drop(["Momentum risk x size", "RF P(decay) x size", "Size only (H1 clicks)", "Week-4 score"], errors="ignore").idxmax()
    lines.append(f"{tag}: best ranker by ROC-AUC is '{best}' ({fmt(b.loc[best, 'ROC-AUC'])}); momentum rule {fmt(b.loc['Momentum rule', 'ROC-AUC'])}; "
                 f"Random Forest minus momentum = {b.loc['Random Forest', 'vs momentum']:+.3f} [{b.loc['Random Forest', 'diff CI low']:+.3f}, {b.loc['Random Forest', 'diff CI high']:+.3f}] "
                 f"-> {'the interval excludes 0' if (b.loc['Random Forest', 'diff CI low'] > 0 or b.loc['Random Forest', 'diff CI high'] < 0) else 'the interval includes 0: the data cannot separate the model from the one-line rule'}.")
    lines.append(f"{tag}: Week-4 score ROC-AUC {fmt(b.loc['Week-4 score', 'ROC-AUC'])} (0.5 = no signal); size only {fmt(b.loc['Size only (H1 clicks)', 'ROC-AUC'])}.")
for (tag, m), t in results.items():
    lines.append(f"{tag} {m}: top-10% by 'RF P(decay) x size' captures {t.loc['RF P(decay) x size', 'Lost clicks captured@10%']:.1%} of lost clicks; "
                 f"'Momentum risk x size' {t.loc['Momentum risk x size', 'Lost clicks captured@10%']:.1%}; size only {t.loc['Size only (H1 clicks)', 'Lost clicks captured@10%']:.1%} (random ~ 10%).")
if len(code_validation):
    ok = code_validation[code_validation["Risk ratio"] > 1.1].index.tolist()
    weak = code_validation[code_validation["Risk ratio"] <= 1.1].index.tolist()
    lines.append(f"Reason codes that separate decayers on held-back months (risk ratio > 1.1): {ok or 'none'}; codes that do not: {weak or 'none'}.")
print("\n".join("- " + l for l in lines))

## ML-12 — Demo outline, social post, employer summary

*Numbers below come from the run recorded in `work/outputs/capstone_results.json` (seed 42). If you re-run with different months, update them.*

### 5-minute demo outline

1. **0:00–0:45 — The decision and a mistake.** Which pages should a content team review first? My first model scored 1.0000 because the label was a SQL rule over its own features (leakage by construction). I caught it in an audit and rebuilt the question.
2. **0:45–1:45 — The honest design.** Features from days 1–15 of a month, label from days 16–end. Train on January–April 2026, test on May and June. Clients split so some are never seen in training. The label is "lost clicks beyond the typical page of that month".
3. **1:45–3:15 — Results.** Test A (19 clients, 1,870 pages): Random Forest ROC-AUC 0.704 vs 0.633 for a one-line click-momentum rule; the difference is +0.071, 95% client-bootstrap interval +0.015 to +0.092. The Week-4 opportunity score scored 0.431, below chance, so it should not be used for this decision. On 6 unseen clients the model and the rule cannot be told apart.
4. **3:15–4:15 — The playbook.** Three reason codes separate decayers on held-back months (impressions falling 2.4x, clicks falling 2.0x, position slipping 1.6x risk ratio); two did not and were dropped. The top-10% queue holds about 40% of lost clicks, but 35–38% of lost clicks still sit in the "no action" tier.
5. **4:15–5:00 — Limits and next step.** Observational data, small client counts, proxy label. Next: more months per client and recorded refresh events, which is what a causal test would need.

### Social-post cut

> My first model scored 1.0000. That was the bug: the label was computed from its own features. I rebuilt it: predict which pages lose clicks in the second half of a month using only the first half, tested on months the model never saw. A Random Forest beat a one-line momentum rule modestly (ROC-AUC 0.70 vs 0.63, 19 clients); the opportunity score I'd built earlier scored below chance (0.43). Observational data, directional only. Paper and code: https://aidib-portfolio.vercel.app/paper/

### Employer-facing summary (3 sentences)

I built an end-to-end, leakage-checked analysis of real search-performance data (FlyRank's pseudonymized ML Internship warehouse): about 22,900 page-months across six months, 5,800 of them above the 50-click floor, with 8 to 25 clients per month. I found that my first model's perfect score was target leakage, redesigned the validation as a time-forward, client-aware test with a deliberate-leak control, and showed that a Random Forest ranks decaying pages modestly better than a one-line rule (ROC-AUC 0.70 vs 0.63, difference interval +0.02 to +0.09). I turned it into a ranked review queue with reason codes validated on held-back months, and reported openly what it misses and what the data cannot support.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
- [ ] Numbers in `work/capstone_report.md` and in the paper match `work/outputs/capstone_results.json` from a fresh run